# KASA-42 — H200 runbook

GPU window: **Thu 30 Jul 2026, 08:32 GMT → Sat 1 Aug, 08:32 GMT**.
Files are **permanently wiped** at window end — no backup, no grace period.
Export to HF Hub as you go, not at the end.

This notebook is a driver, not a codebase. Everything lives in `src/kasa42/`
so the kernel dying does not cost you any work. If a cell fails, fix the `.py`
file, re-run the import cell, and continue.

Order matters. Do not skip the smoke run to save 20 minutes.

The `H+n` headings are hours from window start (08:32 GMT), not from whenever
you happen to be reading this.

## H+0 · Land

The dataset is pre-staged at `/data/ghana-speech` (42 configs, HF sharded
parquet). **Nothing needs downloading** — not the 222 GB, not one shard.
Confirm the layout before trusting it.

In [ ]:
!nvidia-smi
!df -h /data . | tail -3
# Confirm the pre-staged layout: expect <config>/*.parquet, 42 config dirs.
!ls /data/ghana-speech | head -45
!echo '--- configs:' $(ls -d /data/ghana-speech/*/ 2>/dev/null | wc -l)
!ls /data/ghana-speech/Kusaal_kus | head -3

In [ ]:
import os, sys, subprocess
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

!git clone https://github.com/NasamuAlhassan/kasa42.git || (cd kasa42 && git pull --ff-only)
%cd kasa42
!pip install -q -e '.[train,serve]'

# `!python -m kasa42...` runs in a subprocess, where the kernel's sys.path does
# not apply. The editable install above handles it; PYTHONPATH is the backup.
os.environ['PYTHONPATH'] = os.path.join(os.getcwd(), 'src')
print(subprocess.run([sys.executable, '-c', 'import kasa42; print("subprocess import ok")'],
                     capture_output=True, text=True).stdout.strip())

In [ ]:
import sys, torch
sys.path.insert(0, 'src')
print(torch.__version__, torch.cuda.is_available())
print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

### Data pipeline

`manifest.parquet`, `splits.json`, `vocab.json` and `mixture.json` are **not**
in the repo — only `results/manifest_parts/` for 2 of 42 configs, from a
laptop run that never finished over the network.

Build them here instead. `--local-root` reads the pre-staged shards straight
off disk with column projection (metadata columns only, never the audio
column), so the step that was network-bound and hours long is now disk-bound
and minutes long. Existing parts in `results/manifest_parts/` are reused.

Then point everything downstream at `/data/ghana-speech` — there is no local
copy to make.

In [ ]:
GS = '/data/ghana-speech'

# Metadata only: ~1.4 M rows, no audio bytes read. Local disk, so workers can
# go well above the 6 that was safe against the Hub.
!python -m kasa42.data.build_manifest --local-root {GS} --workers 16

!python -m kasa42.data.splits
!python -m kasa42.data.vocab
!python -m kasa42.data.mixture --alpha 0.5 --cap-hours 40

import json, pathlib
for f in ['manifest.parquet', 'splits.json', 'vocab.json', 'mixture.json']:
    p = pathlib.Path('results') / f
    print(f'{f:20s}', f'{p.stat().st_size/1e6:8.1f} MB' if p.exists() else 'MISSING')

## H+1 · Smoke run

30 steps on 5 languages. Proves the loop before you commit hours to it.

In [ ]:
from kasa42.asr.train import train, TrainConfig
train(TrainConfig(smoke=True, data_dir=GS,
                  languages=['Kusaal_kus','Asante_Twi_twi','Ewe_ewe','Dagaare_dga','Mampruli_maw'],
                  out_dir='checkpoints/smoke'))

## H+2 · Test sets, then baselines

Build both evaluation sets first — they are what every later number is scored
on, and `baselines.py` cannot run without them.

* **honest** — segments from books held out of training entirely.
* **leaked** — random segments from books the model *did* train on, minus every
  id the mixture actually used.

Both contain only utterances the model has never seen. The single difference is
whether the *book* was seen, which is exactly the variable under test. Sampling
utterances the model trained on would measure memorisation instead, and that is
a much weaker claim.

Baselines run before the long training run on purpose. If training disappoints
you still have a story; if it succeeds the numbers are already on one axis.

In [ ]:
!python -m kasa42.asr.testset --data-dir {GS} --max-per-config 200

# Check the summary before trusting the leak table: any config listed under
# leak_excluded had too small a leaked pool, because the mixture consumed
# nearly all of its train-book segments.
import json
s = json.load(open('results/testset_summary.json'))
print(s['honest_total'], 'honest /', s['leaked_total'], 'leaked utts')
print('excluded from leak table:', s['leak_excluded'] or 'none')

!python -m kasa42.asr.baselines --which dondo mms whisper --out-dir results/baselines

## H+4 · Main ASR run (~4–7 h)

In [ ]:
train(TrainConfig(max_steps=12000, batch_duration=320.0, data_dir=GS,
                  out_dir='checkpoints/kasa42-asr'))

## H+11 · Kusaal TTS

**Listen to `data/tts/check/*.wav` before starting the fine-tune.** If those
clips are not the same voice, stop and spend the hours on ASR instead.

In [ ]:
!python -m kasa42.tts.prepare --config Kusaal_kus --data-dir {GS}
import IPython.display as ipd, glob
for f in sorted(glob.glob('data/tts/check/*.wav')):
    print(f); ipd.display(ipd.Audio(f))

In [ ]:
!python -m kasa42.tts.finetune --epochs 60

## H+15 · Evaluate — the leaked-vs-honest table is the headline

`evaluate.py` scores the honest set, then the leaked set, then compares them on
the **intersection** of languages present in both — comparing two micro-averages
taken over different language sets would not be like-for-like.

In [ ]:
!python -m kasa42.asr.evaluate --checkpoint checkpoints/kasa42-asr/final.pt

# TTS round-trip: synthesise Kusaal, transcribe it back with the ASR model.
!python -m kasa42.tts.roundtrip

## H+23 · Export — do not leave this to the end

The GPU disappears Sat 08:31 GMT. If judging is after that, this cell **is**
the submission.

In [ ]:
!python -m kasa42.asr.export --checkpoint checkpoints/kasa42-asr/final.pt
# Then verify with the GPU hidden — this is the check that matters:
!CUDA_VISIBLE_DEVICES='' KASA42_MODE=onnx python -c "\
import sys; sys.path.insert(0,'src'); from kasa42.app.app import Engine; \
import numpy as np, time; e=Engine('onnx'); \
t,l,c,dt=e.transcribe(16000, np.zeros(16000,dtype=np.float32)); \
print('mode', e.mode, '|', l, f'{dt*1000:.0f}ms')"

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
# api.upload_folder(folder_path='export', repo_id='PrinceAlhassanNasamu/kasa42-asr', repo_type='model')
# api.upload_folder(folder_path='app',    repo_id='PrinceAlhassanNasamu/kasa42',     repo_type='space')